In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np      
import pandas as pd
from scipy.integrate import complex_ode

from tasks import make_static_sin_task
from encoding_masking import build_masked_input, compute_tdm_params
from Reservoirs.ELM_static import run_elm_static, ELMStaticParams
from readout import split_states_targets, fit_ridge_readout, predict_readout
from metrics import mse, nrmse
from Reservoirs.LangKobayashi import expand_virtual_nodes, simulate_lk, simulate_lk_mini, extract_R_from_E, intensities_after_theta_times, charge_at_theta_intervals
from MC_test import linear_memory_curve, plot_memory_curve

In [2]:
#mask_types = ["binary", "continuous", "m_sequence", "two_sine"]
mask_types = ["constant"]
mask_seeds = [0, 1, 2, 3, 4]
#mask_seeds = [1]
#max_delays = [10,20]
max_delays = [10]

N = 20 
eta = 0.001

In [3]:
#constant paramters

ridge_alpha = 1e-8

TLk = 150
kappa = 0.1  
alpha = 0 
phi = 0 
xi = 0 
D_noise = 10**-7*0
p= 0.05
dt = 0.1 #ONLY VARY AT THE VERY END
theta = 22
tau_factor = 1.41 

washout = 200
#L = 500 + washout #ONLY VARY AT THE VERY END
task_seed = 0
order = 10 #narma order
split = 0.8 # test/predict split

In [4]:
TDM_parameters = compute_tdm_params(N, theta, dt, tau_factor)

tap_stride = TDM_parameters.tap_stride
Nd_loop = TDM_parameters.Nd_loop 
Nd_delay = TDM_parameters.Nd_delay    
tau = TDM_parameters.tau
print(TDM_parameters)

TDMParams(N=20, theta=22, dt=0.1, T=440, tau=620.4, Nd_loop=4400, Nd_delay=6204, tap_stride=220)


In [5]:
L = 1000 
rng = np.random.default_rng(0)
u = rng.uniform(-1.0, 1.0, size=L) #input

In [6]:
results=[]

for mask_type in mask_types:
        for mask_seed in mask_seeds:

            x_norm, mask, V = build_masked_input(u, N=N, rng=mask_seed, mask_type= mask_type)
            Vs = expand_virtual_nodes(V, tap_stride, Nd_loop)

            # run reservoir > get states R
            E_hist, n_hist = simulate_lk(Vs, dt, Nd_delay, alpha=alpha, kappa=kappa, phi=phi, p=p, eta=eta, D_noise=D_noise, xi=xi, Tlk=TLk, E0=1e-3+0j, n0=0.0)
            R = extract_R_from_E(E_hist, L=L, N=N, tap_stride=tap_stride, Nd_loop=Nd_loop, washout_cycles = 0)
            
            # train readout > predict y_pred

            #u = u[washout:]
            for max_delay in max_delays:
                
                delays, capacities, nrmse, MC_est = linear_memory_curve(R, u, max_delay=max_delay, washout=washout, split=split, ridge_alpha=ridge_alpha)
                
                results.append({"mask_type": mask_type, "mask_seed": mask_seed, "N": N, "eta": eta, "ridge_alpha": ridge_alpha, "max_delay": max_delay, "MC_est": MC_est})




df = pd.DataFrame(results)
#df.to_csv("MemoryCapacity_sweep_N50_results.csv", index=False)
#df.to_csv("MemoryCapacity_sweep_N20_results_binary.csv", index=False)


In [7]:
df

# print("Estimated linear MC:", MC_est)

,mask_type,mask_seed,N,eta,ridge_alpha,max_delay,MC_est
0,constant,0,20,0.001,1.000000e-08,10,8.218443
1,constant,1,20,0.001,1.000000e-08,10,8.218443
2,constant,2,20,0.001,1.000000e-08,10,8.218443
3,constant,3,20,0.001,1.000000e-08,10,8.218443
4,constant,4,20,0.001,1.000000e-08,10,8.218443
